The goal of this notebook is to derive metrics indicating the 3D pattern of the spatial predictions.

For this matter, the probabilities are binarized using the Otsu algorithm as to distinguish foreground vs background voxels, where foreground voxels indicate the regions in which a cell type is likely to be located.

Afterwards, the DBSCAN clustering algorithm is applied to the x, y, z coordinates of the foreground voxels as to estimate in how many hot spots a cluster is located, how voluminous those hot spots are and how much spatial scatter is in the predictions as evidenced by high counts of DBSCAN clusters as well as outlier points. 

By using an epsilon of 0.1, new spatial DBSCAN clusters form if they are at least 100 micrometer away from other hot spots. 

A secondary goal is to obtain the center coordinates for the DBSCAN clusters as the center coordinates of the clusters accounting for the highest relative mass will serve as a prior coordinate for a spatially-informed UMAP created later.

In [1]:
import warnings
import json

from pathlib import Path

import pandas as pd
import scanpy as sc
import numpy as np
import zarr

from dotenv import dotenv_values
from matplotlib import pyplot as plt
from skimage.filters import threshold_otsu
from sklearn.cluster import DBSCAN


from hm3d_utils.spatial import BoundingBox3D

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, dpi=120, figsize=(5, 5))

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

SPATIALDIR = DATADIR / "7_spatial"

OUTDIR = SPATIALDIR / "analyses"
OUTDIR.mkdir(exist_ok=True, parents=True)

In [3]:
EPS = 0.1
MIN_SAMPLES = 3

In [4]:
# Load voxel grid Zarr store
voxel_store = zarr.open(SPATIALDIR / 'sp_voxels.zarr')
voxel_store.tree()

/
├── L1_mass (6, 168, 144, 107) float32
├── L1_prob (6, 168, 144, 107) float32
├── L1_rel_mass (6, 168, 144, 107) float32
├── L2_mass (82, 168, 144, 107) float32
├── L2_prob (82, 168, 144, 107) float32
├── L2_rel_mass (82, 168, 144, 107) float32
├── L3_mass (274, 168, 144, 107) float32
├── L3_prob (274, 168, 144, 107) float32
├── L3_rel_mass (274, 168, 144, 107) float32
├── L4_mass (716, 168, 144, 107) float32
├── L4_prob (716, 168, 144, 107) float32
├── L4_rel_mass (716, 168, 144, 107) float32
├── L5_mass (932, 168, 144, 107) float32
├── L5_prob (932, 168, 144, 107) float32
├── L5_rel_mass (932, 168, 144, 107) float32
├── counts (168, 144, 107) float32
├── evidence (168, 144, 107) float32
├── latent_embedding (45, 168, 144, 107) float32
├── region_labels (168, 144, 107) uint32
└── region_masks (176, 168, 144, 107) bool

In [5]:
bbox = BoundingBox3D.load(SPATIALDIR / "bounding_box_ccf_mm.json")
bbox

Bounding Box Ranges (CCFv3 mm):
 x: 4.375<->8.55
 y: 3.95<->7.525
 z: 3.05<->5.7

In [6]:
xx = np.arange(start=bbox.min[0], stop=bbox.max[0], step=0.025) 
yy = np.arange(start=bbox.min[1], stop=bbox.max[1]+0.025, step=0.025) 
zz = np.arange(start=bbox.min[2], stop=bbox.max[2], step=0.025) 

len(xx), len(yy), len(zz)

(168, 144, 107)

In [7]:
# Create 3D meshgrid from coordinate arrays
X, Y, Z = np.meshgrid(xx, yy, zz, indexing='ij')

# Stack into a single array of shape (nx, ny, nz, 3)
grid = np.stack([X, Y, Z], axis=-1)

coords = grid.reshape(-1, 3)

In [8]:
coords

array([[4.375, 3.95 , 3.05 ],
       [4.375, 3.95 , 3.075],
       [4.375, 3.95 , 3.1  ],
       ...,
       [8.55 , 7.525, 5.65 ],
       [8.55 , 7.525, 5.675],
       [8.55 , 7.525, 5.7  ]], shape=(2588544, 3))

In [9]:
def get_DBSCAN_clusters(voxel_store, coords, lvl: int, eps: float, min_samples: int) -> pd.DataFrame:
    """
    Run DBSCAN clustering on voxel probabilities for each celltype at a given hierarchy level.
    
    Args:
        voxel_store:  zarr/h5 store containing relative mass arrays of shape (n_celltypes, nx, ny, nz)
        coords:       voxel coordinates of shape (nx*ny*nz, 3)
        lvl:          hierarchy level to process
        eps:          DBSCAN epsilon (neighborhood radius)
        min_samples:  DBSCAN minimum samples per cluster
    
    Returns:
        pd.DataFrame with columns: cluster_name, otsu_thr, dbscan_label, 
                                   num_vox, percent_vox, center_coords, rel_mass
    """
    # init lists to collect data for a long df
    cluster_list = []
    thr_list = []
    label_list = []
    label_count_list = []
    label_count_relative_list = []
    center_coords_list = []
    rm_list = []

    celltypes = voxel_store[f"L{lvl}_rel_mass"].attrs["labels"]

    for i, celltype in enumerate(celltypes):

        probs = voxel_store[f"L{lvl}_prob"][i, :, :, :].reshape(-1)
        probs = np.nan_to_num(probs, nan=0.0)
        relmass = voxel_store[f"L{lvl}_rel_mass"][i, :, :, :].reshape(-1)
        relmass = np.nan_to_num(relmass, nan=0.0)

        try:
            thr = threshold_otsu(probs)
            mask = probs > thr 

            dat = pd.DataFrame(data=coords[mask, :], columns=["x", "y", "z"])
            dat["rel_mass"] = relmass[mask]
            total_vox = len(dat)

            db = DBSCAN(eps=eps, min_samples=min_samples).fit(dat[["x", "y", "z"]].values)
            labels = list(db.labels_)
            dat["labels"] = labels

            for label in dat["labels"].unique():
                label_mask = dat["labels"] == label
                count = labels.count(label)
                center = dat[label_mask][["x", "y", "z"]].mean().values

                cluster_list.append(celltype)
                thr_list.append(thr)
                label_list.append(label)
                label_count_list.append(count)
                label_count_relative_list.append(count / total_vox)
                center_coords_list.append(center)
                rm_list.append(dat[label_mask]["rel_mass"].sum())

        except Exception as e:
            print(f"Skipping celltype '{celltype}' at level {lvl}: {e}")
            cluster_list.append(celltype)
            thr_list.append(np.nan)
            label_list.append(np.nan)
            label_count_list.append(np.nan)
            label_count_relative_list.append(np.nan)
            center_coords_list.append([np.nan, np.nan, np.nan])
            rm_list.append(np.nan)

    df = pd.DataFrame.from_dict({
        "cluster_name": cluster_list,
        "otsu_thr": thr_list,
        "dbscan_label": label_list,
        "num_vox": label_count_list,
        "percent_vox": label_count_relative_list,
        "center_coords": center_coords_list,
        "rel_mass": rm_list
    })

    return df

In [ ]:
dfs=[]
for lvl in range(1,6):
    df = get_DBSCAN_clusters(voxel_store, coords, lvl = lvl, eps = EPS, min_samples = MIN_SAMPLES)
    dfs.append(df)
db_metrics = pd.concat(dfs, axis=0)

In [14]:
db_metrics

,cluster_name,otsu_thr,dbscan_label,num_vox,percent_vox,center_coords,rel_mass
0,Gb1-2 GABA,0.360830,0,655476,0.960278,"[5.148483277801901, 5.528230972606613, 4.14191...",0.618039
1,Gb1-2 GABA,0.360830,-1,14,0.000021,"[6.700000000000033, 5.789285714285707, 4.26964...",0.000008
2,Gb1-2 GABA,0.360830,1,28,0.000041,"[5.336607142857157, 6.912499999999987, 5.06785...",0.000016
3,Gb1-2 GABA,0.360830,2,10,0.000015,"[5.507500000000017, 5.642499999999994, 3.93499...",0.000005
4,Gb1-2 GABA,0.360830,3,17,0.000025,"[5.498529411764722, 6.20294117647058, 4.248529...",0.000008
...,...,...,...,...,...,...,...
41157,NN5-46 Ependymal_2-1,0.040327,1,9184,0.595977,"[5.5504518728223395, 4.362954594947711, 5.1256...",0.330236
41158,NN5-46 Ependymal_2-1,0.040327,2,61,0.003958,"[5.554098360655756, 5.359016393442618, 5.67909...",0.001370
41159,NN5-46 Ependymal_2-1,0.040327,-1,2,0.000130,"[5.500000000000016, 5.6124999999999945, 5.6999...",0.000040
41160,NN5-46 Ependymal_2-1,0.040327,3,4563,0.296106,"[7.296148367302202, 5.606158229235152, 5.66777...",0.137433


In [15]:
db_metrics.to_csv(OUTDIR / "dbscan_metrics.csv", index=False)

### Let's extract the L2/L3 centroids for a spatially-informed umap

In [16]:
for i in [2,3]:

    # filter for L2/L3 clusters 
    filtered = db_metrics[db_metrics["cluster_name"].str[2] == str(i)]

    # exclude noise label -1
    filtered = filtered[filtered["dbscan_label"] != -1]

    # for each cluster_name, get the row with the highest rel_mass
    top_clusters = filtered.loc[filtered.groupby("cluster_name")["rel_mass"].idxmax()]

    # extract centroid x, y, z rounded to 3 decimals
    top_clusters = top_clusters.copy()
    top_clusters["x"] = top_clusters["center_coords"].apply(lambda c: round(c[0], 3))
    top_clusters["y"] = top_clusters["center_coords"].apply(lambda c: round(c[1], 3))
    top_clusters["z"] = top_clusters["center_coords"].apply(lambda c: round(c[2], 3))

    # final output
    result = top_clusters[["cluster_name", "rel_mass", "x", "y", "z"]]
    result.to_csv(OUTDIR / f"L{i}_umap_cluster_centroids.csv")
